# Fine-tune VLA policies and the ACT baseline on Panthera-HT simulation data

Input: raw episodes from `tools/twin/twin_record.py`, attached as a Kaggle dataset (folder with `episode_*/`, `lerobot_convert.py` and `policy_bridge.py`). `tools/train/kaggle_overnight.sh` uploads the data and starts this notebook unattended.

Without an attached dataset the notebook builds three synthetic episodes and trains each model for a few steps. This checks the pipeline (conversion, training, packaging) without simulator data.

Settings: GPU T4 x2 (one GPU is used), Internet on. All models run in one environment with LeRobot at commit `e581c1d` (Python 3.12, installed with uv), which contains SmolVLA, Evo-1 and ACT. The T4 supports neither bfloat16 nor flash attention, so Evo-1 trains in float16 without them.

Every run saves a checkpoint every `SAVE_FREQ` steps, so a run cut short by the 12-hour session limit still leaves usable checkpoints. Logs go to `/kaggle/working/<model>.log`.

In [ ]:
# Parameters
MODELS = ["smolvla", "act"]          # add "evo1" for the model comparison (ADR 0005)
STEPS = {"smolvla": 15000, "evo1": 15000, "act": 20000}
BATCH = {"smolvla": 16, "evo1": 8, "act": 8}
SAVE_FREQ = 2500
SYNTHETIC_STEPS = 20                 # used when no dataset is attached
LEROBOT = "lerobot[training,evo1,smolvla] @ git+https://github.com/huggingface/lerobot@e581c1d5701f846788662079d4ac08835c96def9"


In [ ]:
# 1) Install LeRobot in a Python 3.12 environment and check the GPU
import subprocess
subprocess.run("pip install -q uv && uv venv -q /kaggle/working/venv --python 3.12", shell=True, check=True)
subprocess.run(["uv", "pip", "install", "-q", "--python", "/kaggle/working/venv/bin/python", LEROBOT], check=True)
VENV = "/kaggle/working/venv/bin"
print(subprocess.run([f"{VENV}/python", "-c", "import torch, lerobot; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'lerobot', lerobot.__version__)"], capture_output=True, text=True).stdout)


In [ ]:
# 2) Locate the raw dataset, or build synthetic episodes
import glob, json, urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
from PIL import Image

candidates = [Path(p).parent for p in glob.glob("/kaggle/input/**/lerobot_convert.py", recursive=True)]
SYNTHETIC = not candidates
if SYNTHETIC:
    RAW = Path("/kaggle/working/synthetic_raw")
    rng = np.random.default_rng(0)
    tasks = [("put the small red cube in the left tray", "en"), ("đặt khối trụ đỏ vào khay tròn", "vi"), ("move a cube to the deep bin", "en")]
    for e, (task, lang) in enumerate(tasks):
        folder = RAW / f"episode_{e:06d}"
        T = 40
        states = rng.normal(size=(T, 7)).astype(np.float32)
        for cam in ("wrist", "workspace"):
            (folder / cam).mkdir(parents=True, exist_ok=True)
            for i in range(T):
                Image.fromarray(rng.integers(0, 255, (240, 320, 3), dtype=np.uint8)).save(folder / cam / f"{i:06d}.jpg")
        np.save(folder / "states.npy", states)
        np.save(folder / "actions.npy", np.roll(states, -1, 0))
        np.save(folder / "timestamps.npy", np.arange(T, dtype=np.float32) / 15)
        (folder / "meta.json").write_text(json.dumps({"task": task, "language": lang, "success": True, "fps": 15, "frames": T, "state_dim": 7, "cameras": ["workspace", "wrist"], "image_size": [320, 240], "format": 1}, ensure_ascii=False))
    url = "https://raw.githubusercontent.com/Nguyen-Hao-Khang/vla-robot-sorting/main/tools/twin/lerobot_convert.py"
    urllib.request.urlretrieve(url, RAW / "lerobot_convert.py")
    STEPS = {k: SYNTHETIC_STEPS for k in STEPS}
    SAVE_FREQ = SYNTHETIC_STEPS
else:
    RAW = candidates[0]
DATA = Path("/kaggle/working/panthera_twin")
REPO = "local/panthera_twin"
metas = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(RAW.glob("episode_*/meta.json"))]
print("raw folder:", RAW, "(synthetic)" if SYNTHETIC else "")
print("episodes:", len(metas), "successful:", sum(m.get("success", False) for m in metas))
print("languages:", Counter(m.get("language", "en") for m in metas))
print("command forms:", Counter(m.get("command_kind", "n/a") for m in metas))


In [ ]:
# 3) Convert to a LeRobot dataset
subprocess.run(["rm", "-rf", str(DATA)], check=True)
result = subprocess.run([f"{VENV}/python", str(RAW / "lerobot_convert.py"), "--raw", str(RAW), "--root", str(DATA), "--repo-id", REPO], capture_output=True, text=True)
print("\n".join((result.stdout + result.stderr).splitlines()[-5:]))
assert result.returncode == 0, "conversion failed"


In [ ]:
# 4) Train every model in MODELS
import time

COMMON = [f"--dataset.repo_id={REPO}", f"--dataset.root={DATA}", "--policy.device=cuda",
          "--policy.push_to_hub=false", f"--save_freq={SAVE_FREQ}", "--log_freq=100", "--num_workers=2"]
POLICY = {
    # smolvla_base expects camera1..camera3: wrist -> camera1, workspace -> camera2, camera3 empty.
    "smolvla": ["--policy.path=lerobot/smolvla_base", "--policy.empty_cameras=1",
                '--rename_map={"observation.images.wrist": "observation.images.camera1", "observation.images.workspace": "observation.images.camera2"}'],
    "evo1": ["--policy.type=evo1", "--policy.dtype=float16", "--policy.use_amp=false", "--policy.use_flash_attn=false"],
    "act": ["--policy.type=act"],
}
summary = []
for model in MODELS:
    out = f"/kaggle/working/train_{model}"
    subprocess.run(["rm", "-rf", out])
    cmd = [f"{VENV}/lerobot-train", *POLICY[model], *COMMON, f"--batch_size={BATCH[model]}",
           f"--steps={STEPS[model]}", f"--output_dir={out}", f"--job_name=panthera_twin_{model}"]
    start = time.time()
    with open(f"/kaggle/working/{model}.log", "w") as log:
        code = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT).returncode
    lines = open(f"/kaggle/working/{model}.log", errors="replace").read().splitlines()
    steps = [l for l in lines if "step:" in l and "loss:" in l]
    summary.append({"model": model, "exit": code, "minutes": round((time.time() - start) / 60, 1),
                    "last_log": steps[-1].split("ot_train.py")[-1].strip() if steps else None})
    print(summary[-1])
    if code:
        print("\n".join(lines[-20:]))
json.dump(summary, open("/kaggle/working/summary.json", "w"), indent=2)


## Package the last checkpoints (Output tab)

In [ ]:
%%bash
cd /kaggle/working
for d in train_*; do
  [ -d $d/checkpoints/last/pretrained_model ] && tar czf ${d}_last.tgz -C $d/checkpoints/last pretrained_model
done
ls -lh /kaggle/working/*.tgz /kaggle/working/*.log /kaggle/working/summary.json


## Run a checkpoint in the simulator (Ubuntu workstation)

1. LeRobot environment: `python tools/twin/policy_bridge.py serve --checkpoint <unpacked>/pretrained_model` (add `--camera-map wrist=camera1,workspace=camera2` for SmolVLA).
2. RoboTwin environment: `python tools/twin/twin_eval.py run --executor policy --checkpoint <label> --episodes 30 --seed 9 --language vi --out results/<label>_vi.jsonl`.